In [1]:
!pip install groq python-dotenv numpy tqdm datasets

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     ---------------------------------------- 90.6/90.6 kB 5.4 MB/s eta 0:00:00
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached charset_normalizer-3.4.4-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
     ---------------------------------------- 0.0/77.6 kB ? eta -:--:--
     ---------------------------------------- 77.6/77.6 kB 4.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/138.3 kB ? eta -:--:--
   ---------------------------------------- 138.3/138.3 kB 4.1 MB/s eta 0:00:00
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
   -----


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

c:\Users\82109\Desktop\YBIGTA_newbie_assignment\ybigta\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [3]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello! I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular topic?


#### GSM8K 데이터셋 확인해보기

In [4]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [5]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, response, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    if len(results) == 0:
        additional_regex = r"\$?([0-9,]+)"
        additional_matches = re.finditer(additional_regex, response, re.MULTILINE)
        results.extend([match.group(1).replace(",", "") for match in additional_matches])

    return results[-1] if results else None

In [6]:
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant", # 모델 이름도 최신으로 유지
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    # tqdm으로 진행 상황 표시
    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        # 정답에서 숫자만 추출
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            
            predicted_answer = extract_final_answer(response)

            # --- [수정된 부분] 안전하게 변환하기 ---
            if isinstance(predicted_answer, str):
                try:
                    # 쉼표 제거 후 빈 문자열인지 확인
                    cleaned_answer = predicted_answer.replace(",", "").strip()
                    if cleaned_answer: # 내용이 있을 때만 변환
                        predicted_answer = float(cleaned_answer)
                    else: # 비어있으면 오답 처리
                        predicted_answer = None 
                except ValueError:
                    # 변환 중 에러 나면(예: 문자가 섞임) 오답 처리
                    predicted_answer = None
            # --------------------------------------
            
            # None이 아니고 오차가 매우 작으면 정답 처리
            if predicted_answer is not None:
                diff = abs(predicted_answer - correct_answer)
                is_correct = diff < 1e-5
            else:
                is_correct = False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [7]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [8]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [9]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:02<00:02,  2.19it/s]

Progress: [5/10]
Current Acc.: [60.00%]


100%|██████████| 10/10 [00:04<00:00,  2.14it/s]

Progress: [10/10]
Current Acc.: [50.00%]


In [15]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!

# Direct Prompting: 0, 3, 5 shot 자동 실행 및 저장

shot_counts = [0, 3, 5]

for shot in shot_counts:
    print(f"========== [Direct Prompting] {shot}-shot Start ==========")
    
    # 1. 프롬프트 생성 (공장 가동)
    prompt = construct_direct_prompt(num_examples=shot)
    
    # 2. 벤치마크 테스트 실행 (50문제 풀기)
    # 주의: 위에서 run_benchmark_test의 model 기본값을 최신 버전으로 바꿨다면 model 인자는 생략해도 됩니다.
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        num_samples=50,  # 과제 요구사항: 50문제
        VERBOSE=False    # 출력 너무 길어지지 않게 끔
    )
    
    # 3. 결과 파일 저장
    filename = f"direct_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"Done! Saved to '{filename}' (Accuracy: {accuracy:.2%})\n")

print("모든 Direct Prompting 테스트가 완료되었습니다.")

========== [Direct Prompting] 0-shot Start ==========


 10%|█         | 5/50 [00:02<00:20,  2.16it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:04<00:21,  1.87it/s]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [40/50]
Current Acc.: [82.50%]


 90%|█████████ | 45/50 [03:42<00:24,  4.81s/it]

Progress: [45/50]
Current Acc.: [80.00%]


100%|██████████| 50/50 [04:14<00:00,  5.09s/it]

Progress: [50/50]
Current Acc.: [80.00%]
Done! Saved to 'direct_prompting_5.txt' (Accuracy: 80.00%)

모든 Direct Prompting 테스트가 완료되었습니다.


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [11]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    prompt = "Instruction:\nSolve the following mathematical question. Think step-by-step to reach the answer.\n"  #TODO: 프롬프트를 작성해주세요!

    for i in range(num_examples):
        idx = sampled_indices[i]
        cur_question = train_dataset['question'][idx]
        cur_answer = train_dataset['answer'][idx] # <--- 핵심! 풀이 과정을 자르지 않고 그대로 가져옵니다.

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:\n{cur_answer}\n"
        
    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [12]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!

# CoT Prompting: 0, 3, 5 shot 자동 실행 및 저장

shot_counts = [0, 3, 5]

for shot in shot_counts:
    print(f"========== [CoT Prompting] {shot}-shot Start ==========")
    
    # 1. CoT 프롬프트 생성 (방금 만든 함수 사용)
    prompt = construct_CoT_prompt(num_examples=shot)
    
    # 2. 벤치마크 테스트 실행 (50문제)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        num_samples=50,
        VERBOSE=False
    )
    
    # 3. 결과 파일 저장 (CoT_prompting_숫자.txt)
    filename = f"CoT_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"Done! Saved to '{filename}' (Accuracy: {accuracy:.2%})\n")

print("모든 CoT Prompting 테스트가 완료되었습니다.")

========== [CoT Prompting] 0-shot Start ==========


 10%|█         | 5/50 [00:02<00:26,  1.70it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:07<00:35,  1.14it/s]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [72.50%]


 90%|█████████ | 45/50 [08:43<01:00, 12.19s/it]

Progress: [45/50]
Current Acc.: [73.33%]


100%|██████████| 50/50 [09:41<00:00, 11.63s/it]

Progress: [50/50]
Current Acc.: [74.00%]
Done! Saved to 'CoT_prompting_5.txt' (Accuracy: 74.00%)

모든 CoT Prompting 테스트가 완료되었습니다.


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [24]:
def construct_my_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train
    
    # 전략 변경: 0-shot일 때 모델이 가장 잘 알아듣는 '심플하지만 강력한' 지시로 변경
    # 복잡한 1,2,3,4 단계 강요보다는 "단계별로 생각해(Chain of Thought)"가 훨씬 효과적입니다.
    prompt = """You are an expert Mathematician.
Solve the following math problem.
Your goal is to provides a step-by-step logical reasoning to reach the correct answer.

IMPORTANT:
- Think step by step.
- Double-check your calculations.
- You MUST end your response with the final answer in exactly this format: "Answer: #### [Number]"
"""

    # Few-shot 예시 추가
    if num_examples > 0:
        sampled_indices = random.sample(
            [i for i in range(len(train_dataset['question']))],
            num_examples
        )
        for i in range(num_examples):
            idx = sampled_indices[i]
            cur_question = train_dataset['question'][idx]
            cur_answer = train_dataset['answer'][idx]
            
            # [수정 포인트] cur_answer에는 이미 '#### 숫자'가 들어있습니다.
            # 따라서 굳이 뒤에 Answer: #### ...를 또 붙일 필요가 없습니다.
            # 대신 모델이 Answer: 태그를 배우도록 '####' 앞부분을 'Answer: ####'로 살짝만 가공해줍니다.
            
            # GSM8K 데이터 원본이 "풀이... #### 숫자" 형태이므로, 
            # 이를 "Reasoning: 풀이... \nAnswer: #### 숫자" 형태로 깔끔하게 정리
            parts = cur_answer.split("####")
            reasoning = parts[0].strip()
            final_val = parts[1].strip()

            prompt += f"\n[Example {i+1}]\n"
            prompt += f"Question: {cur_question}\n"
            prompt += f"Reasoning: {reasoning}\n"
            prompt += f"Answer: #### {final_val}\n"

    # 실제 문제
    prompt += "\nNow, solve this problem step by step:\n"
    prompt += "Question: {question}\n"
    prompt += "Reasoning:\n" # 모델이 생각을 시작하도록 유도

    return prompt

In [ ]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!

# My Prompting: 0, 3, 5 shot 자동 실행 및 저장

shot_counts = [0,3,5]

for shot in shot_counts:
    print(f"========== [My Prompting] {shot}-shot Start ==========")
    
    # 1. 나만의 프롬프트 생성
    prompt = construct_my_prompt(num_examples=shot)
    
    # 2. 벤치마크 테스트 실행 (50문제)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        num_samples=50,
        VERBOSE=False
    )
    
    # 3. 결과 파일 저장 (My_prompting_숫자.txt)
    filename = f"My_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"Done! Saved to '{filename}' (Accuracy: {accuracy:.2%})\n")

print("축하합니다! 모든 과제 실행이 완료되었습니다.")

### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!